In [1]:
# Run these small tutorials on CPU; select a GPU before importing JAX for larger runs.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np
import marlax as mx
from marlax.envs import CooperativeGrid, GridState
from marlax.models import QTable
from marlax.methods import QLearning, SARSA, ExpectedSARSA


In [2]:
# 1. Compare bootstrap rules on the same hand-computable transition.
# Next values are [2, 4], reward is 1, gamma is .5, alpha is .5.
from dataclasses import replace
agent = mx.Agent(QTable())
spec = mx.EnvSpec(1, 2, 2, "two_state_example")
current = mx.PolicyInput(jnp.array([[0]]), jnp.ones((1, 1, 2), dtype=bool))
following = mx.PolicyInput(jnp.array([[1]]), jnp.ones((1, 1, 2), dtype=bool))
batch = mx.Transition(current, jnp.array([[0]]), jnp.array([[1.0]]), following,
                      jnp.array([False]), jnp.array([True]), jnp.array([[0]]))
results = {}
for method_type in (QLearning, SARSA, ExpectedSARSA):
    method = method_type(alpha=0.5, gamma=0.5, epsilon_start=0.2, epsilon_end=0.2)
    learning = method.init(jax.random.PRNGKey(0), agent, spec)
    learning = replace(learning, policy_state=learning.policy_state.at[0, 1].set(jnp.array([2., 4.])))
    updated, _ = method.update(learning, batch, jax.random.PRNGKey(1))
    results[method_type.__name__] = float(updated.policy_state[0, 0, 0])
np.testing.assert_allclose(list(results.values()), [1.5, 1.0, 1.45])
results


{'QLearning': 1.5, 'SARSA': 1.0, 'ExpectedSARSA': 1.4500000476837158}

In [3]:
# 2. Natural termination removes the bootstrap; truncation above retained it.
terminal = batch._replace(terminated=jnp.array([True]))
updated, _ = method.update(learning, terminal, jax.random.PRNGKey(2))
assert updated.policy_state[0, 0, 0] == 0.5
{"terminal_update": float(updated.policy_state[0, 0, 0])}


{'terminal_update': 0.5}

In [4]:
# 3. Train each method for 8,192 transitions using the same API.
# The runner retains SARSA's next action between updates.
env = CooperativeGrid(size=3)
params = env.default_params()._replace(time_limit=16)
agent = mx.Agent(QTable())
summaries = {}
for method_type in (QLearning, SARSA, ExpectedSARSA):
    run = mx.train(env, params, agent, method_type(epsilon_steps=8192),
                   seed=0, num_envs=64, env_steps=8192, chunk_steps=128)
    summaries[method_type.__name__] = {"env_steps": int(run.state.method_state.env_steps),
                                    "table_shape": run.policy.variables.shape}
summaries


{'QLearning': {'env_steps': 8192, 'table_shape': (2, 567, 5)},
 'SARSA': {'env_steps': 8192, 'table_shape': (2, 567, 5)},
 'ExpectedSARSA': {'env_steps': 8192, 'table_shape': (2, 567, 5)}}

In [5]:
# 4. Exploration is scheduled in world transitions, not vector calls or episodes.
method = QLearning(epsilon_steps=8192)
{steps: float(method.epsilon(steps)) for steps in (0, 4096, 8192, 16384)}


{0: 1.0,
 4096: 0.5099999904632568,
 8192: 0.019999980926513672,
 16384: 0.019999980926513672}